# Meeting Transcriber on a Colab GPU

In Colab, select a GPU under **Runtime → Change runtime type**.

1. Upload a recording to `My Drive/meeting-transcriber/recordings/`.
2. Upload the current `meeting-transcriber.zip` to `My Drive/meeting-transcriber/`. If you use a hosted Git repo, set `REPO_URL` to a clone URL with the `colab-drive` branch.
3. Run **Set up the runtime** and approve the Drive connection. Then set `RECORDING` and run **Transcribe a recording**.

Colab copies the recording into its runtime for GPU processing. TXT, SRT, and JSON results are saved to Drive.

In [ ]:
# Set up the runtime
REPO_URL = ""  # Leave empty to clone from the ZIP in Drive.
REPO_BRANCH = "colab-drive"
DRIVE_FOLDER = "meeting-transcriber"
ARCHIVE_NAME = "meeting-transcriber.zip"
ENABLE_SPEAKERS = False

import os
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from google.colab import drive

def run(command, *, env=None):
    with subprocess.Popen(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

gpu_command = ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]
if not shutil.which(gpu_command[0]):
    raise RuntimeError("Select a GPU under Runtime → Change runtime type and run setup again.")
gpu = subprocess.run(gpu_command, capture_output=True, text=True)
if gpu.returncode or not gpu.stdout.strip():
    raise RuntimeError("Colab did not assign an NVIDIA GPU. Connect to a GPU runtime and retry.")
print(f"Colab GPU: {gpu.stdout.strip()}")

drive.mount("/content/drive")
drive_project = Path("/content/drive/MyDrive") / DRIVE_FOLDER
runtime = Path("/content/meeting-transcriber-runtime")
runtime.mkdir(exist_ok=True)
repo = runtime / "repo"
clone = ["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_BRANCH]

if (repo / ".git").is_dir():
    print(f"Using the checkout at {repo}. Start a new runtime to change the repository source.")
elif REPO_URL.strip():
    run([*clone, "--", REPO_URL.strip(), str(repo)])
else:
    archive_path = drive_project / ARCHIVE_NAME
    if not archive_path.is_file():
        raise FileNotFoundError(f"Upload {archive_path.name} to {drive_project} or set REPO_URL.")
    with TemporaryDirectory(prefix="repo-source-", dir=runtime) as source:
        with ZipFile(archive_path) as archive:
            archive.extractall(source)
        checkout = Path(source) / "meeting-transcriber"
        run([*clone, "--no-local", "--", str(checkout), str(repo)])

launcher = repo / "scripts" / "run_gpu.sh"
if not launcher.is_file():
    raise FileNotFoundError("Start a fresh Colab runtime and clone the updated colab-drive branch.")
run(["git", "-C", str(repo), "log", "-1", "--oneline"])

venv = runtime / "venv"
venv_python = venv / "bin" / "python"
if not venv_python.is_file():
    run([sys.executable, "-m", "venv", "--without-pip", str(venv)])
pip = [sys.executable, "-m", "pip", "--python", str(venv_python),
       "install", "--progress-bar", "off"]
extras = "gpu"
if ENABLE_SPEAKERS:
    if not shutil.which("ffmpeg"):
        run(["apt-get", "-qq", "update"])
        run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
    run(pip + [
        "--index-url", "https://download.pytorch.org/whl/cpu",
        "torch==2.8.0", "torchaudio==2.8.0",
    ])
    extras = "gpu,speakers"
run(pip + ["-e", f"{repo}[{extras}]"])

## Transcribe a recording

Set `RECORDING` relative to the project folder in Drive. Use a different `OUTPUT_FOLDER` for a second pass on the same filename. Use `large-v3` for the default GPU run, or try `medium.en` or `small.en` if GPU memory is tight. Model downloads can take a few minutes in a new runtime.

For speaker labels, [accept the speaker model terms](https://huggingface.co/pyannote/speaker-diarization-community-1), create a [Hugging Face read token](https://huggingface.co/settings/tokens), and add an `HF_TOKEN` entry in Colab **Secrets** with notebook access enabled. Set `ENABLE_SPEAKERS = True` and rerun setup. If you know the speaker count, enter it in `SPEAKERS`.

In [ ]:
RECORDING = "recordings/weekly-sync.mp4"
MODEL = "large-v3"
OUTPUT_FOLDER = "transcripts"
HOTWORDS = ""
SPEAKERS = None

recording = drive_project / RECORDING
if not recording.is_file():
    raise FileNotFoundError(f"Recording not found at {recording}. Update RECORDING.")

output_dir = drive_project / OUTPUT_FOLDER
env = dict(os.environ, MEETING_TRANSCRIBER_PYTHON=str(venv_python))
options = ["--model", MODEL, "--output-dir", str(output_dir)]
if HOTWORDS.strip():
    options += ["--hotwords", HOTWORDS.strip()]
if ENABLE_SPEAKERS:
    from google.colab import userdata

    env["HF_TOKEN"] = userdata.get("HF_TOKEN")
    options.append("--diarize")
    if SPEAKERS is not None:
        options += ["--speakers", str(SPEAKERS)]

with TemporaryDirectory(prefix="recording-", dir=runtime) as folder:
    local_recording = Path(folder) / recording.name
    print(f"Copying {recording.name} from Google Drive...", flush=True)
    shutil.copyfile(recording, local_recording)
    run(["bash", str(launcher), str(local_recording), *options], env=env)

print(f"Transcripts are available in {output_dir} on Google Drive.")